# Implementação da LeNet-5 com PyTorch

Conforme solicitado, este notebook contém apenas a declaração da arquitetura clássica LeNet-5 utilizando a biblioteca PyTorch e TorchVision.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        # Camada Conv 1: 1 canal de entrada (grayscale), 6 canais de saída, kernel 5x5
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        
        # Camada Conv 2: 6 canais de entrada, 16 canais de saída, kernel 5x5
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        
        # Camadas Fully Connected (Densas)
        # A saída da conv2 terá 16 canais de 5x5 (supondo imagem de entrada 32x32)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # Conv1 -> ReLU -> MaxPool2d (kernel 2x2, stride 2)
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        
        # Conv2 -> ReLU -> MaxPool2d (kernel 2x2, stride 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        
        # Transformando (flattening) para entrar nas camadas densas
        # Achata os dados a partir da dimensão 1 (ignorando o batch_size na dimensão 0)
        x = torch.flatten(x, 1) 
        
        # Dense 1 -> ReLU
        x = F.relu(self.fc1(x))
        
        # Dense 2 -> ReLU
        x = F.relu(self.fc2(x))
        
        # Saída (Logits)
        x = self.fc3(x)
        
        return x

# Instanciando o modelo para confirmação
model = LeNet5()
print(model)

## Visualizando os Filtros Iniciais

Abaixo podemos visualizar o estado inicial dos pesos (filtros) da primeira camada convolucional (`conv1`) logo após a inicialização aleatória da rede e antes de qualquer processo de treinamento.

In [ ]:
# Instalação das bibliotecas necessárias no kernel atual (caso não existam)
%pip install matplotlib torchvision

In [ ]:
import matplotlib.pyplot as plt

# Extraindo os pesos da camada conv1
# O formato dos pesos na conv1 é [out_channels, in_channels, kernel_height, kernel_width]
# Para a LeNet-5, temos [6, 1, 5, 5]
filters = model.conv1.weight.data.clone().numpy()

fig, axes = plt.subplots(1, 6, figsize=(15, 3))

for i in range(6):
    # Selecionamos o filtro i, com o canal 0 (pois só há 1 canal de entrada)
    ax = axes[i]
    ax.imshow(filters[i, 0], cmap='gray')
    ax.set_title(f'Filtro {i+1}')
    ax.axis('off')

plt.suptitle('Estado inicial dos pesos da Conv1 (Antes do Treinamento)', fontsize=16)
plt.show()

## Importando e Visualizando o Dataset MNIST

A LeNet-5 foi projetada originalmente para reconhecer dígitos manuscritos. Vamos preparar o dataset MNIST e visualizar algumas de suas imagens.

In [ ]:
# Transformação para converter as imagens em tensores e normalizá-las
transform = transforms.Compose([
    transforms.Resize((32, 32)), # LeNet-5 requer imagens 32x32
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Carregando o dataset de treino do MNIST
train_dataset = torchvision.datasets.MNIST(root='./data', 
                                           train=True, 
                                           transform=transform, 
                                           download=True)

# Usando o DataLoader para criar lotes (batches)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, 
                                           batch_size=6, 
                                           shuffle=True)

# Pegando um batch de imagens e rótulos
dataiter = iter(train_loader)
images, labels = next(dataiter)

# Configurando a figura do matplotlib
fig, axes = plt.subplots(1, 6, figsize=(12, 3))

for i in range(6):
    ax = axes[i]
    # Desnormalizando a imagem para visualização (img * std + mean)
    img = images[i][0].numpy() * 0.5 + 0.5
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Rótulo: {labels[i].item()}')
    ax.axis('off')

plt.suptitle('Exemplos de Imagens do MNIST', fontsize=16)
plt.show()

## Treinamento do Modelo

Agora vamos definir a função de custo (Cross Entropy Loss) e o otimizador (Adam ou SGD) para iniciar o treinamento da rede neural LeNet-5 com os dados de treinamento.

In [ ]:
import torch.optim as optim

# Definimos a Função de Perda e o Otimizador
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Definir dispositivo de hardware (GPU ou CPU)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Treinando no dispositivo: {device}")
model.to(device)

# Loop de Treinamento
epochs = 5
print("Iniciando o treinamento...")
for epoch in range(epochs):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        # Obter os inputs; data é uma lista de [inputs, labels]
        inputs, labels = data[0].to(device), data[1].to(device)

        # Zerar as derivadas/gradientes
        optimizer.zero_grad()

        # Forward (caminho de ida) + backward (caminho de volta para calcular os gradientes) + optimize (atualizar pesos)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Imprimir estátisticas no final de vários batchs
        running_loss += loss.item()
        if i % 2000 == 1999:    # printar a cada 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Treinamento Finalizado!')